# Runtime-scaling scan planner

Generates a midpoint-convention BubbleMaster GPU scan from three physical
inputs: the gamma_* range and the kR_* range and N_omega. Grid spacing
(`dgamma_ij=0.4`, `dt=2`) and the resolution/numerics (`wall_points=20`,
`N_min=128`, `panels_per_oscillation=40`, `how_often_ds=5`,
`s_batch_size=128`) are locked to the values validated in
`14_high_gamma_convergence_check.ipynb` -- not exposed as scan inputs here.

Re-running this notebook with a different `(GAMMA_STAR_MIN, GAMMA_STAR_MAX,
KR_MIN, KR_MAX, N_OMEGA)` writes a separate, distinctly-named scan under
`data/runtime_scan_.../` and `scripts/runtime_scan_...sh` -- so several scans
(e.g. increasing `GAMMA_STAR_MAX`) can be run and timed side by side to see
how total runtime scales with the domain size.

**Extending an existing scan**: set `EXTEND_FROM` to an existing
`runtime_scan_.../` directory and raise `GAMMA_STAR_MAX` above what it was
built with (up to `OMEGA_MIN_GAMMA_FACTOR` times the old value -- the
frequency-grid headroom baked in when it was built). This reads the old
scan's actual gamma_ij grid and locked parameters back, extends every
existing row's `times[]` in place to the new, larger `T_MAX`
(`ptbuilder.ic.extend_2d_setup_times` -- wlist/geometry untouched), adds any
new `gamma_ij` rows the larger range needs, and renames the scan directory
to match the new `GAMMA_STAR_MAX`. Re-running `bubblemaster_gpu` on the
(now-extended) setups auto-detects and extends each row from its own
embedded plateau checkpoint -- no separate resume step needed.

In [1]:
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup, extend_2d_setup_times, collision_wall_width
from ptbuilder.weight_scan_diagnostics import load_kinematics

ROOT = Path(pt.__file__).resolve().parent.parent
LAMBDA_BAR = 0.84

## Scan inputs

Only these three define the physical scan domain.

In [2]:
GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 9.6
KR_MIN = 1.0
KR_MAX = 100.0
N_OMEGA = 32

# Optional overrides -- production defaults, only change these deliberately.
PAIR_GAMMA_FACTOR   = 2.5   # gamma_ij,max = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
TIME_FACTOR         = 1.4   # T_MAX = TIME_FACTOR * R_*(GAMMA_STAR_MAX)
OMEGA_MIN_GAMMA_FACTOR = 1.5   # OMEGA_MIN anchored to R_*(this * GAMMA_STAR_MAX),
                               # giving headroom to later extend GAMMA_STAR_MAX by
                               # up to this factor without an empty omega range

# Set to an existing runtime_scan_.../ directory to EXTEND that scan to the
# (larger) GAMMA_STAR_MAX above, instead of building a fresh scan. When set,
# GAMMA_STAR_MIN/KR_MIN/KR_MAX/PAIR_GAMMA_FACTOR/TIME_FACTOR/
# OMEGA_MIN_GAMMA_FACTOR are read back from the old scan and OVERRIDE the
# values above -- only GAMMA_STAR_MAX (the new target) is actually used from
# this cell. Leave as None for a normal, from-scratch scan.
EXTEND_FROM = None

## Locked grid spacing and numerics

Not scan inputs -- these are the production values validated elsewhere in
this repo (`dgamma_ij`/`dt` in `13_reconstruction_scan_resolution.ipynb`;
`N_min`/`panels_per_oscillation`/`wall_points`/`how_often_ds` up to
gamma_ij=24 in `14_high_gamma_convergence_check.ipynb`).

In [3]:
DGAMMA_IJ = 0.4
DT        = 2.0
N_K       = 51

LOCKED = dict(n_min=128, ppw=40, wall_points=20, how_often_ds=5, s_batch_size=128)
CUTOFF_TYPE = 0
BABY_STEPS  = 100
GPU_ACCOUNT = 'm3166_g'

In [4]:
if not (0 < KR_MIN < KR_MAX):
    raise ValueError('Require 0 < KR_MIN < KR_MAX')
if not (1.0 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX):
    raise ValueError('Require 1 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX')
if N_OMEGA < 2:
    raise ValueError('N_OMEGA must be >= 2')

instanton_path = ROOT / f'data/phi4_lambda_bar{LAMBDA_BAR:g}/instanton.h5'
kin = load_kinematics(instanton_path)

class CachedPhi4Potential:
    def __init__(self, lambda_bar):
        self.params = {'lambda_bar': float(lambda_bar)}
    def to_hdf5(self, group):
        group.attrs.create('type', 'phi4', dtype=h5py.string_dtype(encoding='ascii'))
        group.attrs['phi_false'] = 0.0
        group.attrs['phi_true']  = 1.0
        group.attrs['lambda_bar'] = self.params['lambda_bar']

model = SimpleNamespace(instanton=kin.profile, kinematics=kin,
                        potential=CachedPhi4Potential(LAMBDA_BAR))

def time_at_gamma(gamma):
    if gamma <= 1.0:
        return 0.0
    return brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)

def rstar_at_gamma(gamma):
    t = time_at_gamma(gamma)
    return 2.0 * float(kin.R(t, r='mid'))

# --- If extending an existing scan: read its actual manifest rows (not a
# recomputed grid -- a fresh np.linspace over a larger GAMMA_IJ_MAX would
# silently shift EVERY row's gamma_ij, not just add new ones at the end) and
# the locked scan-definition attrs it was built with, overriding this cell's
# own values with them. Only GAMMA_STAR_MAX (the new target) is taken from
# the cell above. ---
old_rows = None
old_gamma_star_max = None
if EXTEND_FROM is not None:
    EXTEND_FROM = Path(EXTEND_FROM)
    old_rows = []
    with open(EXTEND_FROM / 'manifest.tsv') as f:
        next(f)   # header
        for line in f:
            index, gamma, n_t, setup, name = line.rstrip('\n').split('\t')
            old_rows.append(SimpleNamespace(index=int(index), gamma_ij=float(gamma),
                                            n_t=int(n_t), name=name))
    # setup paths are recomputed after the directory rename below, once the
    # new scan_root/setup_dir exist -- see the write-setups cell.

    with h5py.File(EXTEND_FROM / 'setups' / f'{old_rows[-1].name}.h5', 'r') as f:
        old_gamma_star_max     = float(f.attrs['gamma_star_max'])
        GAMMA_STAR_MIN          = float(f.attrs['gamma_star_min'])
        KR_MIN                  = float(f.attrs['kR_min'])
        KR_MAX                  = float(f.attrs['kR_max'])
        PAIR_GAMMA_FACTOR       = float(f.attrs['pair_gamma_factor'])
        TIME_FACTOR             = float(f.attrs['time_factor'])
        OMEGA_MIN_GAMMA_FACTOR  = float(f.attrs['omega_min_gamma_factor'])
        N_OMEGA                 = int(f.attrs['n_w'])

    if GAMMA_STAR_MAX <= old_gamma_star_max:
        raise ValueError(
            f'GAMMA_STAR_MAX ({GAMMA_STAR_MAX}) must be greater than the scan being '
            f'extended\'s GAMMA_STAR_MAX ({old_gamma_star_max}) to extend it')
    if GAMMA_STAR_MAX > OMEGA_MIN_GAMMA_FACTOR * old_gamma_star_max:
        raise ValueError(
            f'GAMMA_STAR_MAX ({GAMMA_STAR_MAX}) exceeds the old scan\'s frequency '
            f'headroom ({OMEGA_MIN_GAMMA_FACTOR:g}x{old_gamma_star_max:g}='
            f'{OMEGA_MIN_GAMMA_FACTOR*old_gamma_star_max:.3f}) -- extending this far '
            f'would need new low-omega bins, which is not supported yet')

    print(f'Extending {EXTEND_FROM.name}: GAMMA_STAR_MAX {old_gamma_star_max:g} -> '
          f'{GAMMA_STAR_MAX:g}  ({len(old_rows)} existing rows, reusing their own '
          f'GAMMA_STAR_MIN={GAMMA_STAR_MIN:g} KR_MIN={KR_MIN:g} KR_MAX={KR_MAX:g} '
          f'PAIR_GAMMA_FACTOR={PAIR_GAMMA_FACTOR:g} TIME_FACTOR={TIME_FACTOR:g} '
          f'OMEGA_MIN_GAMMA_FACTOR={OMEGA_MIN_GAMMA_FACTOR:g} N_OMEGA={N_OMEGA})')

GAMMA_IJ_MAX = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
R_STAR_MAX   = rstar_at_gamma(GAMMA_STAR_MAX)
T_MAX        = TIME_FACTOR * R_STAR_MAX

# OMEGA_MIN is anchored to a gamma_star OMEGA_MIN_GAMMA_FACTOR times larger than
# GAMMA_STAR_MAX (not GAMMA_STAR_MAX itself), so a future scan that raises
# GAMMA_STAR_MAX by up to that factor can reuse this scan's low-omega bins
# without hitting an empty range. R_*(gamma) is linear in gamma, so this only
# costs a small amount of log-spaced resolution (an extra log10(factor)
# decades on top of the existing span).
R_STAR_OMEGA_ANCHOR = rstar_at_gamma(OMEGA_MIN_GAMMA_FACTOR * GAMMA_STAR_MAX)
OMEGA_MIN    = KR_MIN / R_STAR_OMEGA_ANCHOR

if old_rows is not None:
    old_gamma_ij_grid = np.array([r.gamma_ij for r in old_rows])
    last_old = old_gamma_ij_grid[-1]
    n_new = int(np.ceil((GAMMA_IJ_MAX - last_old) / DGAMMA_IJ))
    new_gamma_ij = last_old + DGAMMA_IJ * np.arange(1, n_new + 1)
    gamma_ij_grid = np.concatenate([old_gamma_ij_grid, new_gamma_ij])
    print(f'gamma_ij: {len(old_gamma_ij_grid)} existing rows up to {last_old:.3f}, '
          f'{len(new_gamma_ij)} new rows {new_gamma_ij[0] if len(new_gamma_ij) else float("nan"):.3f}'
          f' .. {gamma_ij_grid[-1]:.3f}')
else:
    n_gamma = int(np.ceil((GAMMA_IJ_MAX - GAMMA_STAR_MIN) / DGAMMA_IJ)) + 1
    gamma_ij_grid = np.linspace(GAMMA_STAR_MIN, GAMMA_IJ_MAX, n_gamma)
    print(f'gamma_ij: {gamma_ij_grid[0]:.3f} .. {gamma_ij_grid[-1]:.3f}  ({n_gamma} pairs, step {DGAMMA_IJ})')

print(f'R_*(gamma_star_max={GAMMA_STAR_MAX}) = {R_STAR_MAX:.3f}')
print(f'R_*(omega_min anchor={OMEGA_MIN_GAMMA_FACTOR}*{GAMMA_STAR_MAX}={OMEGA_MIN_GAMMA_FACTOR*GAMMA_STAR_MAX:.3f}) = {R_STAR_OMEGA_ANCHOR:.3f}')
print(f'T_MAX = {T_MAX:.3f}   OMEGA_MIN = {OMEGA_MIN:.5f}')

gamma_ij: 1.000 .. 24.000  (59 pairs, step 0.4)
R_*(gamma_star_max=9.6) = 133.145
R_*(omega_min anchor=1.5*9.6=14.400) = 199.726
T_MAX = 186.402   OMEGA_MIN = 0.00501


## Write setups + manifest

Folder name encodes the scan's physical inputs, so different
`(GAMMA_STAR_MIN, GAMMA_STAR_MAX, KR_MIN, KR_MAX, N_OMEGA)` choices land in
separate directories automatically.

In [5]:
tag = (f'lb{LAMBDA_BAR:g}_gs{GAMMA_STAR_MIN:g}-{GAMMA_STAR_MAX:g}'
       f'_kR{KR_MIN:g}-{KR_MAX:g}_nw{N_OMEGA}')
scan_root = ROOT / 'data' / f'runtime_scan_{tag}'

n_old = 0
if old_rows is not None:
    n_old = len(old_rows)
    if scan_root != EXTEND_FROM:
        if scan_root.exists():
            raise FileExistsError(
                f'{scan_root} already exists -- refusing to overwrite it while '
                f'extending {EXTEND_FROM.name}')
        EXTEND_FROM.rename(scan_root)
        print(f'Renamed {EXTEND_FROM.name} -> {scan_root.name}  '
              f'(setups/ and gpu/ move together, so existing result_*.h5 files '
              f'and their plateau checkpoints are untouched)')

setup_dir   = scan_root / 'setups'
output_root = scan_root / 'gpu'
setup_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
(ROOT / 'logs').mkdir(exist_ok=True)

params = BubbleMasterParams(
    dz=None, wall_points=LOCKED['wall_points'],
    n_w=N_OMEGA, n_k=N_K, how_often_ds=LOCKED['how_often_ds'],
    baby_steps=BABY_STEPS, cutoff_type=CUTOFF_TYPE, t_0_scal=1.0,
    collision_radius='mid',
)

manifest_rows = []
for index, gamma_ij in enumerate(gamma_ij_grid):
    name = f'g{gamma_ij:08.3f}'.replace('.', 'p')
    setup_path = setup_dir / f'{name}.h5'

    if index < n_old:
        # Existing row: extend its times[] in place to the new, larger T_MAX
        # -- wlist/geometry/t_cut/t_m/t_max/smax all stay exactly as they
        # were (extend_2d_setup_times never touches them), so the row's
        # already-computed result_*.h5 files (and the plateau checkpoint
        # embedded in the last one) remain valid seeds for the GPU binary's
        # own auto-detection.
        with h5py.File(setup_path, 'r') as f:
            old_last_time = float(f['times'][-1])
            old_n_t = int(f.attrs['n_t'])
        assert old_last_time < T_MAX, (
            f'{setup_path.name}: existing last time {old_last_time:.3f} already >= '
            f'new T_MAX={T_MAX:.3f} -- T_MAX should strictly grow with GAMMA_STAR_MAX')
        n_new_t = int(np.ceil((T_MAX - old_last_time) / DT))
        new_times = np.linspace(old_last_time, T_MAX, n_new_t + 1)[1:]
        extend_2d_setup_times(setup_path, new_times)
        n_t = old_n_t + len(new_times)
    else:
        # New row (gamma_ij beyond the old scan's range, or every row for a
        # from-scratch scan): fresh, exactly as before.
        t_first = time_at_gamma(gamma_ij)
        if t_first >= T_MAX:
            raise RuntimeError(f'Collision time exceeds T_MAX at gamma_ij={gamma_ij}')
        n_t = int(np.ceil((T_MAX - t_first) / DT)) + 1
        times = np.linspace(t_first, T_MAX, n_t)

        smallest_target = max(GAMMA_STAR_MIN, gamma_ij / PAIR_GAMMA_FACTOR)
        omega_max = KR_MAX / rstar_at_gamma(smallest_target)
        if omega_max <= OMEGA_MIN:
            raise RuntimeError(f'Empty omega range at gamma_ij={gamma_ij}')
        omega = np.geomspace(OMEGA_MIN, omega_max, N_OMEGA)

        write_2d_setup(model, gamma_ij, times, setup_path, params, wlist=omega)
        with h5py.File(setup_path, 'a') as handle:
            handle.attrs['gamma_star_min']       = GAMMA_STAR_MIN
            handle.attrs['gamma_star_max']       = GAMMA_STAR_MAX
            handle.attrs['kR_min']               = KR_MIN
            handle.attrs['kR_max']               = KR_MAX
            handle.attrs['pair_gamma_factor']    = PAIR_GAMMA_FACTOR
            handle.attrs['time_factor']          = TIME_FACTOR
            handle.attrs['omega_min_gamma_factor'] = OMEGA_MIN_GAMMA_FACTOR
            handle.attrs['R_star_max']           = R_STAR_MAX

    manifest_rows.append((index, gamma_ij, n_t, setup_path.relative_to(ROOT), name))

manifest = scan_root / 'manifest.tsv'
manifest.write_text(
    'index\tgamma_ij\tn_t\tsetup\tname\n' +
    ''.join(f'{i}\t{g:.12g}\t{nt}\t{s}\t{n}\n' for i, g, nt, s, n in manifest_rows)
)
suffix = f'  ({n_old} extended, {len(manifest_rows)-n_old} new)' if old_rows is not None else ''
print(f'Wrote {len(manifest_rows)} setups under {setup_dir.relative_to(ROOT)}{suffix}')
print(f'Manifest: {manifest.relative_to(ROOT)}')

Wrote 59 setups under data/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32/setups
Manifest: data/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32/manifest.tsv


## Rough walltime estimate

Extrapolates from the two real `gamma_ij=24` timings in
`14_high_gamma_convergence_check.ipynb` (`n_s*n_z`-proportional streaming
cost). This is a single-regime calibration point, not a robust fit -- treat
it as a lower-bound sanity check on `GPU_WALLTIME` below, not a guarantee.
Only the largest `gamma_ij` in the grid matters, since all pairs run
concurrently on separate GPUs.

In [6]:
from ptbuilder.ic import _spatial_step, _two_bubble_ic

_A_SECONDS_PER_NSNZ = (74.06 * 60) / (246104 * 560858)   # calibrated at gamma_ij=24
_SAFETY_FACTOR = 3.0   # this model is a single anchor point -- pad generously

g_max = gamma_ij_grid[-1]
dz_target = _spatial_step(model.instanton, g_max, params)
z, _, d, ds = _two_bubble_ic(model.instanton, g_max, dz_target, 'mid')
n_z_max = len(z)
n_s_max = round(1.2 * d / ds)
est_seconds = _A_SECONDS_PER_NSNZ * n_s_max * n_z_max * _SAFETY_FACTOR

print(f'largest gamma_ij in grid: {g_max:.3f}  (n_s={n_s_max:,}  n_z={n_z_max:,})')
print(f'estimated walltime (with {_SAFETY_FACTOR}x safety margin): '
      f'{est_seconds/3600:.2f} h  -- set GPU_WALLTIME below accordingly')

largest gamma_ij in grid: 24.000  (n_s=560,858  n_z=246,104)
estimated walltime (with 3.0x safety margin): 3.70 h  -- set GPU_WALLTIME below accordingly


## SLURM script

Runtimes across the grid vary by 2-3 orders of magnitude (seconds at low
gamma_ij up to hours near GAMMA_IJ_MAX), so sizing the allocation for all
`n_tasks` at once (one node per 4 tasks, all launched together) leaves most
GPUs idle waiting for the few slow, high-gamma_ij pairs to finish.

Instead this requests a small, fixed-size pool (`GPU_NODES` nodes x 4
GPUs/node) and streams the manifest through it with `xargs -P`: as soon as
one `srun` finishes, the next pending row starts on that freed GPU slot.
Still uses the same `--exclusive --exact` per-GPU packing as
`09_adaptive_bm_scan.ipynb` -- only the scheduling around it changes.

In [7]:
GPU_WALLTIME = '04:00:00'   # override manually if the estimate above says otherwise
GPU_NODES    = 2            # size of the worker pool; concurrency = GPU_NODES * 4

gpus_per_node = 4
n_concurrent  = GPU_NODES * gpus_per_node
n_tasks       = len(manifest_rows)
relative_manifest = manifest.relative_to(ROOT)
relative_output   = output_root.relative_to(ROOT)

# Manifest rows are passed through xargs by ROW NUMBER only, never as raw
# tab-delimited text -- BSD xargs (macOS) silently collapses embedded tabs to
# spaces when substituting into -I, which would empty every field past the
# first. A bare integer has no such ambiguity on any xargs implementation.
script_path = ROOT / 'scripts' / f'runtime_scan_{tag}_gpu.sh'
script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{tag}_gpu
#SBATCH --output=logs/bm_{tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes={GPU_NODES}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={GPU_WALLTIME}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {LOCKED['n_min']} --filon-panels-per-osc {LOCKED['ppw']} --s-batch-size {LOCKED['s_batch_size']}
}}
export -f run_one

seq 1 {n_tasks} | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
script_path.chmod(0o755)
print(f'Written: {script_path.relative_to(ROOT)}  '
      f'({n_tasks} tasks, {n_concurrent} concurrent slots on {GPU_NODES} nodes)')
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')

Written: scripts/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32_gpu.sh  (59 tasks, 8 concurrent slots on 2 nodes)
Submit: sbatch scripts/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32_gpu.sh
